# STATISTICS DELIVERABLE 2

##WHY WE HAVE HAD TO CHANGE THE DATAFRAME?


Given the impossibility of developing linear and logistic regression models due to the large amount of data, the option of clustering was explored using the k-means method. Columns with continuous values were selected and the elbow method was applied to determine an optimal number of clusters. However, this strategy did not produce satisfactory results. In the linear regressions, the assumptions about normality of errors and homoscedasticity were not met, with R² values close to 0 and inappropriate patterns in the graphs. In the logistic regressions, the Hosmer-Lemeshow and McNemar tests showed low p-values, suggesting that there was no significant difference between random classification and model prediction.

In [ ]:
packages <- c("tidyverse","fastDummies","psych",'stats','class',"urltools","GGally", "ggrepel",'car',
              'lmtest','gvlma','performance','tseries','caret','mlbench','moments','ResourceSelection','nnet','pROC')

newpack  <- packages[!(packages %in% installed.packages()[,"Package"])]
if(length(newpack)) install.packages(newpack)
a<-lapply(packages, library, character.only=TRUE)
Sys.setlocale('LC_ALL','C')

In [ ]:
cardio_train <- read.csv("cardio_train.csv",sep=';')

## DATA REDUCTION

cardio_train <- cardio_train %>%
  mutate('BMI'=weight/(height/100)^2,age_years=round(age/365)) %>%  arrange(age)

##### Data cleaning

outlaiers.detect <- function(vector){
  iqr <- IQR(vector)

  q1 <- quantile(vector, 0.25)
  q3 <- quantile(vector, 0.75)

  # obtain the limits
  limit_lower <- q1 - 1.5 * iqr
  limit_upper <- q3 + 1.5 * iqr

  # Find outliers
  outliers <- vector[vector < limit_lower | vector > limit_upper]
  return(outliers)
}


outlaiers.BMI <- outlaiers.detect(cardio_train$BMI)
outlaiers.AP_LO <- outlaiers.detect(cardio_train$ap_lo)
outlaiers.AP_HI <- outlaiers.detect(cardio_train$ap_hi)

cardio_train <- cardio_train %>% filter(!(BMI %in% outlaiers.BMI | ap_lo %in% outlaiers.AP_LO | ap_hi %in% outlaiers.AP_HI ))
cardio_train <- cardio_train %>% filter(!(age_years==30))
cardio_train
#### FUNCTION CREATED

minmax <- function(x){
  x <- (x-min(x))/(max(x)-min(x))
  return(x)
}

## In order to have a better data manipulation and to do some models, first of all, I will normalize all the values using the min-max normalization to obtain all the data in the same scale
## cardio,cholesterol,gluc,smoke,alco,active this columns are has discrete values which means that I have to create dummy cols in order to be able to use its.


data.for.models<-cardio_train %>% select(age,age_years,gender,BMI,ap_hi,ap_lo,cardio,cholesterol,gluc,smoke,alco,active) %>%
  mutate(across(c(age,BMI,ap_hi,ap_lo),minmax), gender=case_when(gender==1 ~ 0,
                                                                 gender==2 ~1 )) %>% dummy_cols(select_columns = c('cholesterol','gluc'))
data.for.models

In [ ]:
### GENERAL DATA
colnames(data.for.models)
str(data.for.models)
correlation_matrix <- cor(data.for.models) ## ap_hi and ap_lo have a correlation value higher than 0.7
print(correlation_matrix)
corPlot(data.for.models)

In [ ]:
##################
#    clusters.   #
##################


## Furthermore, we still having a lot of data per cluster so we have decided to divide the data using the k-means method,by using the chol/age/gluc to obtain the culsters also we have used the elbow method in order to obtain the most optimal number of it:


set.seed(123)
residuals <- numeric(10) ##vector with length 10 and zeros

for (i in 1:10) {
  model <- kmeans(data.for.models[,c('age_years','cholesterol','gluc')], centers = i)
  residuals[i] <- model$tot.withinss
}

residuals.df<-data.frame('number.of.clusters'=c(1:10),'residuals'=residuals)

ggplot(residuals.df, aes(x = number.of.clusters, y = residuals)) +
  geom_point() +
  geom_line() +  # Add a line connecting points
  labs(x = "Number of Clusters", y = "Residuals") +
  ggtitle("ELBOW METHOD")


set.seed(123)
data.for.models <-data.for.models %>% mutate(group=kmeans(data.for.models[,c('age_years','cholesterol','gluc')], centers = 5)$cluster)


summarise.of.data<-data.for.models %>% group_by(group)%>% summarise(nuber.of.patients=n(),avg.years=mean(age_years),avg.chol=mean(cholesterol),avg.gluc=mean(gluc), percent.cardio.patologies=mean(cardio)*100) %>% arrange(desc(avg.years))
summarise.of.data
corPlot(summarise.of.data)


group1.data <- data.for.models %>% filter(group==1)
group2.data <- data.for.models %>% filter(group==2)
group3.data <- data.for.models %>% filter(group==3)
group4.data <- data.for.models %>% filter(group==4)
group5.data <- data.for.models %>% filter(group==5)

In [ ]:
#######################
#   LINEAR MODEL      #
#######################

####### GENERIC EXAMPLE
model1.1<-lm(ap_hi~age + gender + BMI + ap_lo + cardio + smoke + alco + active + cholesterol_1 + cholesterol_2 + cholesterol_3 + gluc_1 + gluc_2 + gluc_3 ,data=data.for.models)
summary(model1.1)

model1.2<-lm(ap_hi~ age + gender + BMI + ap_lo + cardio + smoke + active + cholesterol_1 + cholesterol_2 +  gluc_1 + gluc_2 ,data=data.for.models)
summary(model1.2)

model1.3<-lm(ap_hi~age + gender + BMI + ap_lo + cardio + smoke + active + cholesterol_1 + gluc_1 + gluc_2,data=data.for.models)
summary(model1.3)

model1.4<-lm(ap_hi~age + gender + BMI + ap_lo + cardio + smoke + active + cholesterol_1 + gluc_2,data=data.for.models)
summary(model1.4)
plot(model1.4)## observing the qqplot of the residuals we can determine that this residuals doesn't follow a normal distribution but using some kind of manipulation we can change this?

compare_performance(model1.1,model1.2,model1.3,model1.4)
m1.4_residuals <- model1.4$residuals
mean(m1.4_residuals)## near to 0


car::vif(model1.4)# no multilinearity
##library(lmtest)
bptest(model1.4) ## using thos test we confirme our hypotesis the model has heterocedasticisty


##library(gvlma)

gvlma(model1.4)

######## group1
modelg1<-lm(ap_hi~age + gender + BMI + ap_lo + cardio + smoke + alco + active + cholesterol_1 + cholesterol_2 + cholesterol_3 + gluc_1 + gluc_2 + gluc_3 ,data=group1.data)
summary(modelg1)

modelg1.2<-lm(ap_hi~gender + BMI + ap_lo + cardio + smoke + alco + active + cholesterol_1 + cholesterol_2 + cholesterol_3 + gluc_1 + gluc_2 + gluc_3 ,data=group1.data)
summary(modelg1.2)

modelg1.3<-lm(ap_hi~gender + BMI + ap_lo + cardio + alco + active + cholesterol_1 + cholesterol_2 + cholesterol_3 + gluc_1 + gluc_2 + gluc_3 ,data=group1.data)
summary(modelg1.3)

modelg1.4<-lm(ap_hi~gender + BMI + ap_lo + cardio + active + cholesterol_1 + cholesterol_2 + cholesterol_3 + gluc_1 + gluc_2 + gluc_3 ,data=group1.data)
summary(modelg1.4)

modelg1.5<-lm(ap_hi~gender + BMI + ap_lo + cardio + active + cholesterol_1 + cholesterol_2 + cholesterol_3 + gluc_2 + gluc_3 ,data=group1.data)
summary(modelg1.5)

modelg1.6<-lm(ap_hi~gender + BMI + ap_lo + cardio + active + cholesterol_1 + cholesterol_2 + cholesterol_3 +  gluc_3 ,data=group1.data)
summary(modelg1.6)

modelg1.7<-lm(ap_hi~gender + BMI + ap_lo + cardio + active + cholesterol_1 + cholesterol_2 + cholesterol_3 ,data=group1.data)
summary(modelg1.7)

modelg1.8<-lm(ap_hi~gender + BMI + ap_lo + cardio + active + cholesterol_1 + cholesterol_2 + cholesterol_3 ,data=group1.data)
summary(modelg1.8)

modelg1.9<-lm(ap_hi~gender + BMI + ap_lo + cardio + active + cholesterol_1 + cholesterol_2 ,data=group1.data)
summary(modelg1.9)

modelg1.10<-lm(ap_hi~gender + BMI + ap_lo + cardio +  cholesterol_1 + cholesterol_2 ,data=group1.data)
summary(modelg1.10)

compare_performance(modelg1,modelg1.2,modelg1.3,modelg1.4,modelg1.5,modelg1.6,modelg1.7,modelg1.8,modelg1.9,modelg1.10)
## best model modelg1

plot(modelg1)


mg1_residuals <- modelg1$residuals
hist(mg1_residuals)
mean(mg1_residuals)

mg1_box <- boxplot(mg1_residuals)
index_outliers<-names(mg1_box$out)
data.filter <- group1.data %>%
  filter(!(row_number() %in% index_outliers))

modelg1 <- lm(ap_hi~age + gender + BMI + ap_lo + cardio + smoke + alco + active + cholesterol_1 + cholesterol_2 + cholesterol_3 + gluc_1 + gluc_2 + gluc_3 ,data=data.filter)
summary(modelg1)

mg1_residuals <- modelg1$residuals
hist(mg1_residuals)

plot(modelg1)
bptest(modelg1)

In [ ]:
###############################
#   LOGISTIC METHOD           #
###############################

# Split the data between training and testing
dim.test <- round(nrow(data.for.models) * 0.10)
index.test <- sample(1:nrow(data.for.models), dim.test)
data.test <- data.for.models[index.test, ]
data.training <- data.for.models[-index.test, ]

# Fit the logistic regression model
model <- glm(cardio ~ age_years + BMI + ap_hi + ap_lo + alco + active + smoke + cholesterol_2 + cholesterol_3 + gluc_1 + gluc_2, data = data.training, family = "binomial")

# Make predictions on the test set and calculate the prediction probability
data.test <- data.test %>%
  mutate(Prob.predict = predict(model, data.test, type = "response")) %>%
  mutate(cardio.predict = ifelse((Prob.predict >= 0.6), 1, 0))

# Calculate the confusion matrix
confusion_matrix <- table(data.test$cardio, data.test$cardio.predict)

# Calculate precision
accuracy <- sum(diag(confusion_matrix)) / sum(confusion_matrix)

# Calculate ROC curve and AUC-ROC
roc_curve <- roc(data.test$cardio, data.test$cardio.predict)
auc <- auc(roc_curve)

# Print the confusion matrix
print(confusionMatrix(as.factor(data.test$cardio), as.factor(data.test$cardio.predict)))

# Calculate the Hosmer-Lemeshow test
hosmer_test <- hoslem.test(data.test$cardio, data.test$Prob.predict)
print(hosmer_test)

#INFORMATION FROM THE NEW DATABASE

**Information about the data frame package studied.**

The data set PimaIndiansDiabetes2 contains a corrected version of the original data set. While the UCI repository index claims that there are no missing values, closer inspection of the data shows several physical impossibilities, e.g., blood pressure or body mass index of 0. In PimaIndiansDiabetes2, all zero values of glucose, pressure, triceps, insulin and mass have been set to NA, see also Wahba et al (1995) and Ripley (1996).

##Libraries and packages to download

In [ ]:
rm(list = ls()) #This command is included to avoid confusion with previously declared variables

In [ ]:
packages <- c("tidyverse","fastDummies","psych",'stats','class',"urltools","GGally", "ggrepel",'car',
              'lmtest','gvlma','performance','tseries','caret','mlbench','moments','ResourceSelection','nnet','pROC','glue')

newpack  <- packages[!(packages %in% installed.packages()[,"Package"])]
if(length(newpack)) install.packages(newpack)
a<-lapply(packages, library, character.only=TRUE)
Sys.setlocale('LC_ALL','C')

# Cargar el conjunto de datos PimaIndiansDiabetes
data("PimaIndiansDiabetes", package = "mlbench")
data <- PimaIndiansDiabetes


##Descriptive statistics


In this part of the code we have focused on finding how each column of our dataframe is distributed using the statistic_description function, which returns a df with all the trend measures, and the use of histograms, boxplots and barplots depending on whether the variable is continuous quantitative or discrete qualitative.

###statistic_description() FUNCTION

In [ ]:
statistic_description <- function(x){
##Measures of central tendency:
  Mean<- mean(x)
  Mode<- as.numeric(names(which.max(table(x))))
  Median<- median(x)

##Measures of dispersion:
  Ranges<-range(x,na.rm=T)[2]-range(x,na.rm=T)[1]
  Standar_variation<- sd(x)
  Variance<- var(x)
  Quartile_0<- quantile(x)[1]
  Quartile_0.25 <- quantile(x)[2]
  Quartile_0.5 <- quantile(x)[3]
  Quartile_0.75 <- quantile(x)[4]
  Quartile_1 <- quantile(x)[5]

## Shape measurements
  Skewness<-moments::skewness(x)
  Kurtosis<- moments::kurtosis(x)
  data <- tibble(Mean,Mode,Median, Ranges,Variance,Standar_variation,Quartile_0,Quartile_0.25,Quartile_0.5,Quartile_0.75,Quartile_1,Skewness,Kurtosis)
  return(data)
}


###Columns with quantitative information

####AGE
Age (years)

In [ ]:
## AGE
age.hist <- ggplot(data,aes(x=age))+
  geom_histogram()
age.hist
age.box <- boxplot(data$age)
statistic_description(data$age)
shapiro.test(data$age)


#### GLUCOSE LEVELS
Plasma glucose concentration (glucose tolerance test)

In [ ]:
## GLUCOSE
glucose.hist <- ggplot(data,aes(x=glucose))+
  geom_histogram()
glucose.hist
glucose.box <- boxplot(data$glucose)
statistic_description(data$glucose)
shapiro.test(data$glucose)

####INSULIN LEVELS
2-Hour serum insulin (mu U/ml)

In [ ]:
## INSULIN
insulin.hist <- ggplot(data,aes(x=insulin))+
  geom_histogram()
insulin.hist
insulin.box <- boxplot(data$insulin)
statistic_description(data$insulin)

####PRESSURE LEVELS
Diastolic blood pressure (mm Hg)

In [ ]:
## PRESSURE
pressure.hist <- ggplot(data,aes(x=pressure))+
  geom_histogram()
pressure.hist
pressure.box <- boxplot(data$pressure)
statistic_description(data$pressure)


####TRICEPS
Triceps skin fold thickness (mm)

In [ ]:
## TRICEPS
triceps.hist <- ggplot(data,aes(x=triceps))+
  geom_histogram()
triceps.hist
triceps.box <- boxplot(data$triceps)
statistic_description(data$triceps)

####MASS LEVEL
Body mass index (weight in kg/(height in m)^2)

In [ ]:
## MASS
mass.hist <- ggplot(data,aes(x=mass))+
  geom_histogram()
mass.hist
mass.box <- boxplot(data$mass)
statistic_description(data$mass)

####PEDIGREE
Diabetes pedigree function

In [ ]:
## PEDIGREE
pedigree.hist <- ggplot(data,aes(x=pedigree))+
  geom_histogram()
pedigree.hist
mass.box <- boxplot(data$pedigree)
statistic_description(data$pedigree)

###Columns with discrete/quantitative and qualitative information

####DIABETES
Class variable (test for diabetes)

In [ ]:
## DIABETES
diabetes.bar <- ggplot(data,aes(x=diabetes))+
  geom_bar()
diabetes.bar

####PREGNANT
Number of times pregnant

In [ ]:
## PREGNANT
pregnant.hist <- ggplot(data,aes(x=pregnant))+
  geom_histogram()
pregnant.hist
pregnant.box <- boxplot(data$pregnant)
statistic_description(data$pregnant)


##Data Transformation

The diabetes variable is a qualitative variable, however it is in a character format. Due to this, in order to be able to use it in the implementation of the statistical models/linear and logistic regression, we must transform it into a dummy column.

In [ ]:
data <- data %>% mutate(diabetes=ifelse(diabetes=='pos',1,0))
summary(data)
head(data)


## Data Exploration/Correlations

The correlation matrix is a table that shows the linear relationships between multiple variables. Each variable is correlated with all the others, indicating whether they increase together (+1), decrease together (-1), or are unrelated (0). It is a crucial tool in data analysis to understand the interdependence between variables. Matrix values help identify patterns and relationships, making it easier to make decisions for statistical models.

In [ ]:
cor(data)
corPlot(data)
pairs.panels(data)


In [ ]:
ggpairs(data,aes( alpha = 0.5),lower = list(continuous = "smooth"))


#   STUDY OF MODELS

##LINEAR MODELS


- compare_performance()


> The `compare_performance(` function calculates model performance indices for different models simultaneously, allowing comparison of these indices between different models. This makes it easy to benchmark the performance of models and helps in selecting the most suitable model for a specific problem. In the case of linear regression we observe the following parameters: AIC (weights) | AICc (weights) | BIC (weights) | R2 | R2 (adj.) | RMSE | Sigma.(explain AIC,BIC,R2)


- shapiro.test()


> The Shapiro-Wilk test `shapiro.test()` evaluates whether a sample of data follows a normal distribution. The null hypothesis is that the data comes from a normal distribution. If the p-value is less than the chosen significance level (usually 0.05), the null hypothesis is rejected, concluding that the data are not normal.


- bptest()


> The `bptest()` function is used to perform the Breusch-Pagan test for heteroscedasticity in a regression model. Heteroskedasticity occurs when the error variance of a regression model is not constant across all observations. The Breusch-Pagan test evaluates whether there is heteroskedasticity in the model residuals. The null hypothesis is that there is no heteroscedasticity, and a small p value indicates that the null hypothesis is rejected, suggesting the presence of heteroskedasticity. This test is useful for verifying one of the key assumptions of linear regression.


- glvma()


> The gvlma() function evaluates whether a linear model meets crucial assumptions such as linearity, homogeneity of variance, normality of residuals, and independence of errors. Provide test statistics for each assumption. If any assumption does not hold significantly, it suggests problems in model fit, which could require additional adjustments.


- plot(model)

> Using plot(model) allows us to evaluate the goodness of fit and the assumptions of the model, thanks to graphs such as residuals vs fitted, scale-location, normal quantiles and residuals vs lever, to diagnose the model.

**Removal of outliers to improve model normalization**
> Throughout these linear regression models you will be able to see how we have used the boxplot to identify outlier errors in a data set, including the residuals of a regression model. By removing these outliers, we can improve the assumption of normality of errors in our regression model. However, while handling outliers is important to ensure the validity and reliability of analysis results, it is crucial to approach this process carefully and consider the potential side effects and limitations associated with removing these values.





###FIRST MODEL-PREDICT THE GLUCOSE LEVEL


> Suitable for use



In [ ]:
## data selection using the backwards method
model1 <- lm(glucose~.,data=data)
summary(model1)

model1.2 <- lm(glucose~pressure+triceps+insulin+mass+pedigree+age+diabetes,data=data)
summary(model1.2)

model1.3 <- lm(glucose~pressure+triceps+insulin+mass+age+diabetes,data=data)
summary(model1.3)

compare_performance(model1,model1.2,model1.3)
##thanks to this table we can choose the model1.3 as the best one due to AIC/R^2 fitted/BIC

m1.3_residuals <- model1.3$residuals
hist(m1.3_residuals)
mean(m1.3_residuals)
shapiro.test(m1.3_residuals)## the errors don't follow a normal distribuition
#due to that we will compute the boxplot and delete the outlier in order to create a normali distribuition of the errors

m1.3_box <- boxplot(m1.3_residuals)
index_outliers<-names(m1.3_box$out)
data.filter <- data %>%
  filter(!(row_number() %in% index_outliers))

model1.3 <- lm(glucose~pressure+triceps+insulin+mass+age+diabetes,data=data.filter)
summary(model1.3)

m1.3_residuals <- model1.3$residuals
hist(m1.3_residuals)
shapiro.test(m1.3_residuals)## aceptable dependiendo del nivel de significancia


bptest(model1.3)
gvlma(model1.3,alphalevel = 0.01)

In [ ]:
plot(model1.3)
confint(model1.3)


###SECOND MODEL-PREDICT THE PRESSURE LEVEL'
> Suitable for use

In [ ]:
model2 <- lm(pressure~.,data=data)
summary(model2)

model2.2 <- lm(pressure~ glucose + triceps + insulin + mass + pedigree + age + diabetes ,data=data)
summary(model2.2)

model2.3 <- lm(pressure~ glucose + triceps + mass + pedigree + age + diabetes ,data=data)
summary(model2.3)

model2.4 <- lm(pressure~ glucose + triceps + mass + pedigree + age + diabetes ,data=data)
summary(model2.4)

compare_performance(model2,model2.2,model2.3,model2.4)
##thanks to this table we can choose the model2.3 as the best one due to AIC/R^2 fitted/BIC

m2.3_residuals <- model2.3$residuals
hist(m2.3_residuals)
mean(m2.3_residuals)
shapiro.test(m2.3_residuals)

m2.3_box <- boxplot(m2.3_residuals)
index_outliers<-names(m2.3_box$out)
data.filter <- data %>%
  filter(!(row_number() %in% index_outliers))

model2.3 <- lm(pressure~ glucose + triceps +  mass  + age + diabetes,data=data.filter)
summary(model2.3)

m2.3_residuals <- model2.3$residuals
hist(m2.3_residuals)
shapiro.test(m2.3_residuals)

bptest(model2.3)
gvlma(model2.3)


In [ ]:
plot(model2.3)
confint(model2.3)

###THIRD MODEL-PREDICT THE TRICEPS LEVEL
> Not suitable for use

In [ ]:
model3 <- lm(triceps~.,data=data)
summary(model3)

model3.2 <- lm(triceps~ glucose + pressure + triceps + mass + pedigree + age + diabetes ,data=data)
summary(model3.2)

model3.3 <- lm(triceps~  pressure + triceps + mass + pedigree + age + diabetes ,data=data)
summary(model3.3)

model3.4 <- lm(triceps~  pressure + triceps + mass + pedigree + age ,data=data)
summary(model3.4)

compare_performance(model3,model3.2,model3.3,model3.4)
##thanks to this table we can choose the model3 as the best one due to AIC/R^2 fitted/BIC

m3_residuals <- model3$residuals
hist(m3_residuals)
mean(m3_residuals)
shapiro.test(m3_residuals)

m3_box <- boxplot(m3_residuals)
index_outliers<-names(m3_box$out)
data.filter <- data %>%
  filter(!(row_number() %in% index_outliers))

model3 <- lm(triceps~.,data=data.filter)
summary(model3)

m3_residuals <- model3$residuals
hist(m3_residuals)
shapiro.test(m3_residuals)

bptest(model3)
gvlma(model3,alphalevel = 0.01)


In [ ]:
plot(model3)

###FOURTH MODEL-PREDICT THE INSULIN LEVEL
>Not suitable for use


In [ ]:

model4 <- lm(insulin~.,data=data)
summary(model4)

model4.2 <- lm(insulin~ glucose + pressure + triceps + mass + pedigree + age + diabetes,data=data)
summary(model4.2)

model4.3 <- lm(insulin~ glucose +  triceps + mass + pedigree + age + diabetes,data=data)
summary(model4.3)

model4.4 <- lm(insulin~ glucose +  triceps + pedigree + age + diabetes,data=data)
summary(model4.4)

model4.5 <- lm(insulin~ glucose +  triceps + pedigree + age ,data=data)
summary(model4.5)

compare_performance(model4,model4.2,model4.3,model4.4,model4.5)
##thanks to this table we can choose the model4.4 as the best one due to AIC/R^2 fitted/BIC

m4.4_residuals <- model4.4$residuals
hist(m4.4_residuals)
mean(m4.4_residuals)
shapiro.test(m4.4_residuals)

m4.4_box <- boxplot(m4.4_residuals)
index_outliers<-names(m4.4_box$out)
data.filter <- data %>%
  filter(!(row_number() %in% index_outliers))

model4.4 <- lm(insulin~ glucose +  triceps + pedigree + age + diabetes,data=data.filter)
summary(model4.4)

m4.4_residuals <- model4.4$residuals
hist(m4.4_residuals)
shapiro.test(m4.4_residuals)

bptest(model4.4)
gvlma(model4.4,alphalevel = 0.01)


In [ ]:
plot(model4.4)

### FIFTH MODEL-PREDICT THE MASS LEVEL
> Not suitable for use

In [ ]:
model5 <- lm(mass~.,data=data)
summary(model5)

model5.2 <- lm(mass~glucose+pressure+triceps+insulin+pedigree+age+diabetes,data=data)
summary(model5.2)

model5.3 <- lm(mass~glucose+pressure+triceps+pedigree+age+diabetes,data=data)
summary(model5.3)

model5.4 <- lm(mass~glucose+pressure+triceps+age+diabetes,data=data)
summary(model5.4)

model5.5 <- lm(mass~glucose+pressure+triceps+diabetes,data=data)
summary(model5.5)

model5.6 <- lm(mass~glucose+triceps+diabetes,data=data)
summary(model5.6)

compare_performance(model5,model5.2,model5.3,model5.4,model5.5,model5.6)
## thanks to this table we can determine that the most usefull-optumus model is the 5.4

m5.4_residuals <- model5.4$residuals
hist(m5.4_residuals)
mean(m5.4_residuals)



m5.4_box <- boxplot(m5.4_residuals)
index_outliers<-names(m5.4_box$out)
data.filter <- data %>%
  filter(!(row_number() %in% index_outliers))

model5.4 <- lm(mass~glucose+pressure+triceps+age+diabetes,data=data.filter)
summary(model5.4)

m5.4_residuals <- model5.4$residuals
hist(m5.4_residuals)
shapiro.test(m5.4_residuals)

bptest(model5.4)
gvlma(model5.4,alphalevel = 0.01)

##Although the result of the GLVMA is negative, we can affirm that the model is usable since although the errors do not follow a strict normal distribution, given the large number of sample-residual values obtained, we can apply the central limit theorem.



In [ ]:
plot(model5.4)


###SIXTH MODEL-PREDICT THE PEDIGREE LEVEL
> Not suitable for use

In [ ]:

model6 <- lm(pedigree~.,data=data)
summary(model6)

model6.2 <- lm(pedigree~pregnant+pressure+triceps+insulin+mass+age+diabetes,data=data)
summary(model6.2)

model6.3 <- lm(pedigree~pregnant+triceps+insulin+mass+age+diabetes,data=data)
summary(model6.3)

model6.4 <- lm(pedigree~pregnant+triceps+insulin+age+diabetes,data=data)
summary(model6.4)

model6.5 <- lm(pedigree~pregnant+triceps+insulin+diabetes,data=data)
summary(model6.5)

model6.6 <- lm(pedigree~triceps+insulin+diabetes,data=data)
summary(model6.6)

compare_performance(model6,model6.2,model6.3,model6.4,model6.5,model6.6) #the r^2 fitted is really near to 0


###SEVENTH MODEL-PREDICT AGE
>Not suitable for use

In [ ]:

model7 <- lm(age~.,data=data)
summary(model7)

model7.2 <- lm(age~pregnant+glucose+pressure+triceps+insulin+mass+pedigree+diabetes,data=data)
summary(model7.2)

model7.3 <- lm(age~pregnant+glucose+pressure+triceps+mass+pedigree+diabetes,data=data)
summary(model7.3)

model7.4 <- lm(age~pregnant+glucose+pressure+triceps+pedigree+diabetes,data=data)
summary(model7.4)

model7.5 <- lm(age~pregnant+glucose+pressure+triceps+diabetes,data=data)
summary(model7.5)

model7.6 <- lm(age~pregnant+glucose+pressure+triceps,data=data)
summary(model7.6)

compare_performance(model7,model7.2,model7.3,model7.4,model7.5,model7.6)
##thanks to this table we can choose the model7.6 as the best one due to AIC/R^2 fitted/BIC

m7.6_residuals <- model7.6$residuals
hist(m7.6_residuals)
mean(m7.6_residuals)
shapiro.test(m7.6_residuals)

m7.6_box <- boxplot(m7.6_residuals)
index_outliers<-names(m7.6_box$out)
data.filter <- data %>%
  filter(!(row_number() %in% index_outliers))

model7.6 <- lm(age~pregnant+glucose+pressure+triceps+insulin+mass+pedigree+diabetes,data=data.filter)
summary(model7.6)

m7.6_residuals <- model7.6$residuals
hist(m7.6_residuals)
shapiro.test(m7.6_residuals)


bptest(model7.6)
gvlma(model7.6)



In [ ]:
plot(model7.6)

##ANALYSIS OF THE PREDICTION OBTAINED

The PimaIndiansDiabetes dataset is a derived version called PimaIndiansDiabetes2. In the original set, PimaIndiansDiabetes2, we find NA values that have been replaced by 0 in PimaIndiansDiabetes1. Therefore, in order to study the behaviors of our models on the PimaIndiansDiabetes2 data set we decided to perform another method of substitution of NA values agreed upon by data science theory where these are replaced by the column mean.

Thanks to this we obtain a set of data of the same dimensions with real values. Thanks to this we carry out the study of the error committed in the regression in relation to the prediction and the real value.

In [ ]:
data("PimaIndiansDiabetes2", package = "mlbench")
raw_data <- PimaIndiansDiabetes2
data.to.predict <- raw_data %>% mutate(diabetes=ifelse(diabetes=='pos',1,0))
index.complete.cases <- which(complete.cases(data.to.predict)==T)

for (col in names(data.to.predict)) {
  col_mean <- mean(data.to.predict[[col]], na.rm = TRUE)
  data.to.predict[[col]][is.na(data.to.predict[[col]])] <- col_mean
}
head(data.to.predict)
head(data)

Previously we have obtained the index of all the rows that did not have any NA, because these would be repeated in the two dataframes. Once removed we have a completely new dataframe where there is no similar row between the PimaIndiansDiabetes2 and PimaIndiansDiabetes data. Given this, we proceed to study our models.

In [ ]:
data.to.predict<-data.to.predict[-index.complete.cases,]
dim(data.to.predict)

It should be noted that we have not divided the original dataframe between test and training due to the small amount of data we have, which would mean a great loss of precision in the linear regression models, an event that does not happen so seriously in other regression models. or machine learning. Besides, creating a dataframe made up of invented patients does not make sense since these random values could be perfectly outliers.

**How do we rate the goodness of the linear regression model?**

The square root of the Residual Sum of Squares, RSS, is a commonly used measure to evaluate the goodness of fit of a regression model. The RSS represents the sum of the squares of the residuals, that is, the sum of the differences between the observed values and the values predicted by the model, squared.

The root mean square of the RSS is often known as the residual standard deviation (RSD) or the residual standard error. This measure provides an estimate of the spread of the residuals around the regression line.

A low RSD indicates that the residuals tend to be close to the regression line, suggesting that the model fits the data well. On the other hand, a high RSD indicates that the residuals are more spread out, suggesting that the model may not fit the data well.

### Study of the First model

In [ ]:
## model1.3
## glucose~pressure+triceps+insulin+mass+age+diabetes
columns.of.interest<-data.to.predict[,c('pressure','triceps','insulin','mass','age','diabetes')]
prediction.values<-predict(model1.3,columns.of.interest)

study<-data.to.predict %>% select(glucose) %>%
                              mutate(predict=prediction.values,residual=(glucose-predict)^2)

RSD<-round(sqrt(mean(study$residual)),3)
glue('An RSD value of {RSD} means that, on average, model predictions can vary by about more or less {RSD} units from the observed values.')

###Study of the Second model

In [ ]:
## model2.3
## pressure~ glucose + triceps + mass + pedigree + age + diabetes
columns.of.interest<-data.to.predict[,c('glucose','triceps','pedigree','mass','age','diabetes')]
prediction.values<-predict(model2.3,columns.of.interest)

study<-data.to.predict %>% select(glucose) %>%
                              mutate(predict=prediction.values,residual=(glucose-predict)^2)

RSD<-round(sqrt(mean(study$residual)),3)
glue('An RSD value of {RSD} means that, on average, model predictions can vary by about more or less {RSD} units from the observed values.')

##LOGISTIC REGRESION MODELS

- confusionMatrix()


> The confusionMatrix() function provides several performance evaluation metrics for classification models. Among these we observe Accuracy, Sensitivity, Specificity, Mcnemar's Test P-Value.
- Mcnemar's Test P-Value.
The p-value of the McNemar test indicates the probability of observing a difference in the proportions of correct and incorrect classification between the two methods, given that the null hypothesis is true. Additionally, another interpretation of this p-value allows us to identify if our classification model differs from a random classification.
- Also another measure worth highlighting are the Accuracy i NIR values since when the precision (accuracy) of a model is greater than the level of no information (no information rate, NIR), it means that the model is making more accurate predictions than simply predict the majority class in all cases.

- ROC()

> The Roc function allows us to observe the (AUC): The AUC is a summary measure of the model's performance and represents the probability that the model correctly classifies a positive random instance higher than a negative random instance. The higher the AUC, the better the performance of the model.

- hoslem.test()



> The Hosmer-Lemeshow test evaluates whether a logistic regression model fits the data well.



###FIRST MODEL
> Suitable for use

In [ ]:
## In order to know several elements about our model we have to split the data in two training and test
set.seed(1234)
dim.test <- round(nrow(data) * 0.10)
index.test <- sample(1:nrow(data), dim.test)
data.test <- data[index.test, ]
data.training <- data[-index.test, ]


logmodel1 <- glm(diabetes~.,data=data.training, family = "binomial")
summary(logmodel1)

logmodel1.2 <- glm(diabetes~pregnant+glucose+pressure+insulin+mass+pedigree+age,data=data.training, family = "binomial")
summary(logmodel1.2)

logmodel1.3 <- glm(diabetes~pregnant+glucose+pressure+mass+pedigree+age,data=data.training, family = "binomial")
summary(logmodel1.3)

logmodel1.4 <- glm(diabetes~pregnant+glucose+pressure+mass+pedigree,data=data.training, family = "binomial")
summary(logmodel1.4)

compare_performance(logmodel1,logmodel1.2,logmodel1.3,logmodel1.4)

## we have decided that our best model is the logmodel1.2

data.test <- data.test %>% mutate(prediction=predict(logmodel1.2,data.test[,-9],type = "response")) %>%
                              mutate(diabetes.prediction=ifelse(prediction>=0.5,1,0))

roc(data.test$diabetes, data.test$diabetes.prediction)##ROC

confusion_Matrix <- table(data.test$diabetes,data.test$diabetes.prediction)
accuracy <- sum(diag(confusion_Matrix)) / sum(confusion_Matrix)

confusionMatrix(as.factor(data.test$diabetes),as.factor(data.test$diabetes.prediction))


hoslem.test(data.training$diabetes, logmodel1.2$fitted.values)



##MULTIPLE LOGISTIC VS KNN


In this segment of the study, we have investigated the distinction between multiple classification models, focusing particularly on KNN and multiple logistic regression, thus obtaining a percentage difference between models. For that we will seek to classify-predict the number of pregnancies of a person

###MULTIPLE LOGISTIC

In [ ]:

set.seed(1234)
dim.test_2 <- round(nrow(data) * 0.10)
index.test_2 <- sample(1:nrow(data), dim.test_2)
data.test_2 <- data[index.test_2, ]
data.training_2<- data[-index.test_2, ]

model.multi <- multinom(pregnant~.,data.training_2)
summary(model.multi)

predicctions <- predict(model.multi, newdata = data.test_2[,-1],type='class')

accuracy.lgmulti <- mean(predicctions == data.test_2$pregnant)
cat("The accuracy of the multinomial model is :", accuracy.lgmulti*100, "\n")
predicctions <- factor(predicctions, levels = levels(data.test_2$pregnant))



###KNN METHOD

In [ ]:
k.neighbors<-round(sqrt(dim(data.training_2)[1])/2) ## thumb_method <- sqrt(n)/2

model.knn <- knn(train=data.training_2[,-1],test=data.test_2[,-1],cl=data.training_2[,1],k=k.neighbors)

accuracy.knn <- sum(model.knn == data.test_2$pregnant) / nrow(data.test_2)
cat("The accuracy level of the k-NN model with k =", k.neighbors, ":", accuracy.knn*100, "\n")

# Convertir las etiquetas de clase en factores con los mismos niveles
data.test_2$pregnant <- factor(data.test_2$pregnant, levels = levels(model.knn))

data.test_2<-data.test_2 %>% mutate(result=(pregnant==model.knn))
data.test_2

In [ ]:
install.packages("plotly")
library(plotly)

###Percentage difference

In [ ]:
accuracies<-c('logistic'=accuracy.lgmulti,'knn'=accuracy.knn)
increment.difference<-((max(accuracies)-min(accuracies))/min(accuracies))*100
cat('The', names(accuracies)[which.max(accuracies)], 'model has a', increment.difference, '% increment in accuracy compared to the', names(accuracies)[which.min(accuracies)], 'model.')